# Notebook 10 — Multi-Pair Portfolio with Equal Risk Contribution

**Adaptive Pair Trading | Ayush Arora (MQMS2404)**

---

## The Key Insight: Diversification Is the Only Free Lunch

A single-pair strategy is fragile:
- The TATASTEEL/HINDALCO relationship may weaken or break
- One adverse event (corporate action, regulatory change) can wipe out a trade
- A single Sharpe of 0.4 is noise; a portfolio of 5 uncorrelated Sharpes of 0.3 is signal

This notebook constructs a **risk-diversified portfolio** of the top pairs from NB08:

1. **Kalman Filter** — adaptive hedge ratio for each pair (removes static-β basis risk)
2. **Signal generation** — independent Z-score strategy per pair
3. **Correlation analysis** — measure inter-pair P&L dependence
4. **Equal Risk Contribution (ERC)** weights — each pair contributes equally to portfolio volatility
5. **Portfolio analytics** — Sharpe, diversification ratio, drawdown, turnover

### What is ERC?
In an equal-weight portfolio, a high-volatility pair dominates risk.
ERC solves this: weights are chosen so that **every pair contributes the same amount
of volatility** to the portfolio — a first-principles risk-parity approach.

For `n` pairs with covariance matrix Σ:
```
min_w  Σᵢ (RC_i − 1/n)²
s.t.   Σᵢ wᵢ = 1,  wᵢ > 0

where RC_i = wᵢ × (Σw)_i / √(wᵀΣw)
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
from scipy.optimize import minimize
import warnings
from config import (
    PRICES_FILE, TOP_PAIRS_FILE,
    ROLL_WINDOW, ROLL_MIN_PERIODS,
    ENTRY_Z, EXIT_Z, TC,
    KF_DELTA, KF_INIT_P,
    PORT_TOP_PAIRS, PORT_MAX_WEIGHT, PORT_MIN_WEIGHT
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

In [ ]:
prices = pd.read_csv(PRICES_FILE, index_col=0, parse_dates=True)

# Load top pairs from NB08; fall back to the primary pair if NB08 not run yet
try:
    top_pairs_df = pd.read_csv(TOP_PAIRS_FILE, index_col=0)
    top_pairs    = list(zip(top_pairs_df['Stock A'], top_pairs_df['Stock B']))
    top_pairs    = [(a, b) for a, b in top_pairs
                    if a in prices.columns and b in prices.columns]
    top_pairs    = top_pairs[:PORT_TOP_PAIRS]
    print(f'Loaded {len(top_pairs)} pairs from NB08 top_pairs.csv')
except FileNotFoundError:
    # Fallback: use the primary pair plus sector-plausible alternatives
    top_pairs = [
        ('TATASTEEL.NS',  'HINDALCO.NS'),
        ('HDFCBANK.NS',   'ICICIBANK.NS'),
        ('TCS.NS',        'INFY.NS'),
        ('ASIANPAINT.NS', 'BERGEPAINT.NS'),
        ('SUNPHARMA.NS',  'CIPLA.NS'),
    ]
    top_pairs = [(a, b) for a, b in top_pairs
                 if a in prices.columns and b in prices.columns]
    print(f'NB08 output not found — using {len(top_pairs)} fallback pairs')

for a, b in top_pairs:
    print(f'  {a.replace(".NS",""):15s} / {b.replace(".NS","")}')

## Section 1: Kalman Filter Adaptive Hedge Ratio for Each Pair

The same Kalman Filter from NB07 is applied to every pair independently.
This eliminates static-β look-ahead bias and allows each relationship to
evolve at its own pace.

In [ ]:
def kalman_hedge(y, x, delta=KF_DELTA):
    """Kalman filter for time-varying [alpha_t, beta_t]. See NB07 for derivation."""
    n     = len(y)
    theta = np.zeros((n, 2))
    P     = np.eye(2) * KF_INIT_P
    Q     = delta / (1 - delta) * np.eye(2)
    R     = 1.0
    kf_e  = np.zeros(n)

    for t in range(n):
        H       = np.array([[1.0, x.iloc[t]]])
        t_pred  = theta[t-1] if t > 0 else np.zeros(2)
        P_pred  = P + Q
        y_hat   = float(H @ t_pred)
        e       = y.iloc[t] - y_hat
        S       = float(H @ P_pred @ H.T) + R
        R       = 0.95 * R + 0.05 * e ** 2
        K       = P_pred @ H.T / S
        theta[t] = t_pred + K.flatten() * e
        P        = (np.eye(2) - K @ H) @ P_pred
        kf_e[t]  = e

    return (
        pd.Series(theta[:, 0], index=y.index, name='kf_alpha'),
        pd.Series(theta[:, 1], index=y.index, name='kf_beta'),
        pd.Series(kf_e,        index=y.index, name='kf_spread'),
    )

print('Kalman filter defined.')

In [ ]:
pair_data = {}
for a, b in top_pairs:
    kf_alpha, kf_beta, kf_spread = kalman_hedge(prices[a], prices[b])
    # Rolling Z-score on Kalman innovations (no look-ahead)
    rm = kf_spread.rolling(ROLL_WINDOW, min_periods=ROLL_MIN_PERIODS).mean()
    rs = kf_spread.rolling(ROLL_WINDOW, min_periods=ROLL_MIN_PERIODS).std()
    kf_zscore = (kf_spread - rm) / (rs + 1e-10)
    label = f'{a.replace(".NS","")} / {b.replace(".NS","")}'
    pair_data[label] = {
        'a': a, 'b': b,
        'kf_beta':   kf_beta,
        'kf_spread': kf_spread,
        'kf_zscore': kf_zscore,
    }
    print(f'{label:35s} β range [{kf_beta.min():.3f}, {kf_beta.max():.3f}]')

print(f'\nKalman filter estimated for {len(pair_data)} pairs.')

## Section 2: Signal Generation & Individual Pair P&L

In [ ]:
def pair_pnl(zscore, spread, tc=TC):
    """Run Z-score strategy; return daily % P&L."""
    sma   = spread.dropna().abs().mean()
    sret  = spread.diff() / (sma + 1e-10)
    pos   = pd.Series(0.0, index=zscore.index)
    curr  = 0.0
    for t in zscore.index:
        z = zscore.loc[t]
        if pd.isna(z):
            pos.loc[t] = 0.0
            continue
        if curr == 0:
            if   z >  ENTRY_Z: curr = -1.0
            elif z < -ENTRY_Z: curr =  1.0
        elif curr ==  1 and z > -EXIT_Z: curr = 0.0
        elif curr == -1 and z <  EXIT_Z: curr = 0.0
        pos.loc[t] = curr
    trade = pos.diff().abs().fillna(0)
    pnl   = pos.shift(1) * sret - trade * tc
    return pnl.fillna(0)

# Compute daily P&L for each pair
pnl_dict = {}
for label, d in pair_data.items():
    pnl_dict[label] = pair_pnl(d['kf_zscore'], d['kf_spread'])

pnl_df = pd.DataFrame(pnl_dict).dropna(how='all')

# Individual pair statistics
print(f'\n{"Pair":<40} {"Ann Return%":>12} {"Sharpe":>8} {"Max DD%":>9}')
print('-' * 73)
for col in pnl_df.columns:
    r       = pnl_df[col]
    ann_r   = r.mean() * 252 * 100
    ann_v   = r.std() * np.sqrt(252)
    sh      = ann_r / 100 / ann_v if ann_v > 0 else np.nan
    cum     = (1 + r).cumprod()
    mdd     = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    print(f'{col:<40} {ann_r:>12.2f} {sh:>8.3f} {mdd:>9.2f}')

## Section 3: Pair P&L Correlation Analysis

The diversification benefit of a multi-pair portfolio depends critically on
how correlated the individual pair P&Ls are. If all pairs make and lose money
together, there is no diversification.

Ideally, cross-sector pairs should have near-zero P&L correlation.

In [ ]:
pnl_corr = pnl_df.corr()
short_labels = [c.replace('.NS','').replace(' / ',' /\n') for c in pnl_df.columns]

fig, ax = plt.subplots(figsize=(max(7, len(pnl_df.columns)+2),
                                max(5, len(pnl_df.columns))))
sns.heatmap(
    pnl_corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
    xticklabels=short_labels, yticklabels=short_labels,
    vmin=-0.4, vmax=0.4, center=0, ax=ax, linewidths=0.5,
    annot_kws={'size': 9}
)
ax.set_title('Pair Strategy P&L Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pair_pnl_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

avg_corr = pnl_corr.values[np.triu_indices_from(pnl_corr.values, k=1)].mean()
print(f'Average pairwise P&L correlation: {avg_corr:.3f}')
print(f'Diversification present: {"YES" if avg_corr < 0.4 else "LIMITED"}')

## Section 4: Equal Risk Contribution (ERC) Portfolio Weights

ERC ensures every pair contributes the same fraction of total portfolio volatility.
This is more sophisticated than equal-weight (naive) and simpler than
full mean-variance optimisation (which is notoriously sensitive to return estimates).

ERC is used by multi-strategy hedge funds precisely because it requires no return
forecasts — only the covariance matrix, which is estimated from historical data.

In [ ]:
def erc_weights(cov, min_w=PORT_MIN_WEIGHT, max_w=PORT_MAX_WEIGHT):
    """Equal Risk Contribution weights via scipy optimisation."""
    n  = len(cov)
    w0 = np.ones(n) / n

    def portfolio_vol(w):
        return np.sqrt(w @ cov @ w)

    def risk_contribution(w):
        vol = portfolio_vol(w)
        mrc = cov @ w
        return w * mrc / (vol + 1e-12)

    def objective(w):
        rc     = risk_contribution(w)
        target = portfolio_vol(w) / n
        return np.sum((rc - target) ** 2)

    result = minimize(
        objective, w0,
        method='SLSQP',
        bounds=[(min_w, max_w)] * n,
        constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
        options={'ftol': 1e-12, 'maxiter': 2000}
    )
    return result.x

# Estimate daily covariance from pair P&Ls
cov_matrix = pnl_df.cov().values
w_erc      = erc_weights(cov_matrix)
w_equal    = np.ones(len(pnl_df.columns)) / len(pnl_df.columns)

print('=== Portfolio Weights ===')
print(f'\n{"Pair":<40} {"Equal-Weight":>13} {"ERC Weight":>12}')
for label, we, wr in zip(pnl_df.columns, w_equal, w_erc):
    print(f'{label:<40} {we*100:>12.1f}% {wr*100:>11.1f}%')

# Verify equal risk contributions
vol_port_erc = np.sqrt(w_erc @ cov_matrix @ w_erc)
rc_erc       = w_erc * (cov_matrix @ w_erc) / vol_port_erc
print(f'\nRisk contribution per pair (ERC):')
for label, rc in zip(pnl_df.columns, rc_erc):
    pct = rc / rc_erc.sum() * 100
    print(f'  {label:<40} {pct:.1f}%')
print(f'Max deviation from equal contribution: {abs(rc_erc/rc_erc.sum() - 1/len(rc_erc)).max()*100:.2f}%')

## Section 5: Portfolio Equity Curve & Analytics

In [ ]:
# Portfolio daily P&L
pnl_equal = pnl_df @ w_equal
pnl_erc   = pnl_df @ w_erc

cum_equal = (1 + pnl_equal).cumprod() - 1
cum_erc   = (1 + pnl_erc  ).cumprod() - 1

# Individual cumulative returns for comparison
cum_individuals = {col: (1 + pnl_df[col]).cumprod() - 1 for col in pnl_df.columns}

fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)

# ── Main equity curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
for label, cum_i in cum_individuals.items():
    short = label.split('/')[0].strip().replace('.NS','')
    ax1.plot(cum_i * 100, linewidth=0.7, alpha=0.5, linestyle='--', label=f'{short} (indiv.)')
ax1.plot(cum_equal * 100, color='steelblue', linewidth=1.8, label='Equal-weight portfolio')
ax1.plot(cum_erc   * 100, color='crimson',   linewidth=2.2, label='ERC portfolio')
ax1.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax1.set_title('Multi-Pair Portfolio Equity Curves (Kalman Adaptive Hedge)', fontweight='bold')
ax1.set_ylabel('Cumulative Return (%)')
ax1.legend(fontsize=8, ncol=2)
ax1.grid(alpha=0.25)

# ── ERC weights pie ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
short_labels_pie = [c.split('/')[0].strip().replace('.NS','') + '/' +
                    c.split('/')[1].strip().replace('.NS','') for c in pnl_df.columns]
ax2.pie(w_erc, labels=short_labels_pie, autopct='%1.1f%%', startangle=90)
ax2.set_title('ERC Portfolio Weights')

# ── Rolling 252d Sharpe ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
roll_sh_erc = pnl_erc.rolling(252).apply(
    lambda r: r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan
)
roll_sh_eq  = pnl_equal.rolling(252).apply(
    lambda r: r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan
)
ax3.plot(roll_sh_erc, color='crimson',  linewidth=1.0, label='ERC')
ax3.plot(roll_sh_eq,  color='steelblue', linewidth=0.8, alpha=0.7, label='Equal-weight')
ax3.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax3.set_title('Rolling 252-day Sharpe Ratio')
ax3.set_ylabel('Sharpe')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.2)

# ── Drawdown ─────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
dd_erc   = (((1+pnl_erc  ).cumprod() - (1+pnl_erc  ).cumprod().cummax()) /
             (1+pnl_erc  ).cumprod().cummax()) * 100
dd_equal = (((1+pnl_equal).cumprod() - (1+pnl_equal).cumprod().cummax()) /
             (1+pnl_equal).cumprod().cummax()) * 100
ax4.fill_between(dd_erc.index,   dd_erc,   0, alpha=0.4, color='crimson',   label='ERC')
ax4.fill_between(dd_equal.index, dd_equal, 0, alpha=0.3, color='steelblue', label='Equal-weight')
ax4.set_title('Drawdown Profile')
ax4.set_ylabel('Drawdown (%)')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.2)

# ── Monthly P&L heatmap for ERC ───────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
monthly = pnl_erc.resample('ME').sum() * 100
monthly_df = monthly.to_frame('pnl')
monthly_df['Year']  = monthly_df.index.year
monthly_df['Month'] = monthly_df.index.month
try:
    pivot = monthly_df.pivot(index='Year', columns='Month', values='pnl')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                ax=ax5, linewidths=0.3, annot_kws={'size': 7})
    ax5.set_title('Monthly P&L % (ERC Portfolio)')
except Exception:
    ax5.set_visible(False)

plt.savefig('portfolio_equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def perf_stats(pnl, label):
    r       = pnl[pnl != 0]
    ann_r   = r.mean() * 252
    ann_v   = r.std()  * np.sqrt(252)
    sharpe  = ann_r / ann_v if ann_v > 0 else np.nan
    cum     = (1 + pnl).cumprod()
    dd      = (cum - cum.cummax()) / cum.cummax()
    max_dd  = dd.min()
    calmar  = ann_r / abs(max_dd) if max_dd != 0 else np.nan
    hit     = (r > 0).mean()
    # Diversification ratio: sum of individual vols / portfolio vol
    return pd.Series({
        'Ann. Return (%)': round(ann_r * 100, 2),
        'Ann. Vol (%)':    round(ann_v * 100, 2),
        'Sharpe':          round(sharpe, 3),
        'Max DD (%)':      round(max_dd * 100, 2),
        'Calmar':          round(calmar, 3),
        'Hit Rate (%)':    round(hit * 100, 1),
    }, name=label)

rows = [perf_stats(pnl_df[c], c.split('/')[0].strip().replace('.NS','') + '/' +
                               c.split('/')[1].strip().replace('.NS',''))
        for c in pnl_df.columns]
rows.append(perf_stats(pnl_equal, 'EQUAL-WEIGHT PORTFOLIO'))
rows.append(perf_stats(pnl_erc,   'ERC PORTFOLIO          '))

summary = pd.DataFrame(rows)

# Diversification ratio
indiv_vols = np.array([pnl_df[c].std() * np.sqrt(252) for c in pnl_df.columns])
port_vol_erc   = pnl_erc.std() * np.sqrt(252)
port_vol_equal = pnl_equal.std() * np.sqrt(252)
div_ratio_erc   = (w_erc   @ indiv_vols) / port_vol_erc
div_ratio_equal = (w_equal @ indiv_vols) / port_vol_equal

print('\n' + '='*70)
print('  FINAL PERFORMANCE SUMMARY')
print('='*70)
print(summary.to_string())
print('='*70)
print(f'\nDiversification Ratio (ERC)          : {div_ratio_erc:.3f}')
print(f'Diversification Ratio (Equal-weight) : {div_ratio_equal:.3f}')
print('(> 1.0 means portfolio vol < weighted avg of individual vols — risk reduced by diversification)')

## Conclusion

### What makes this portfolio construction institutional-grade

| Feature | Naive approach | This project |
|---------|---------------|--------------|
| Hedge ratio | Static OLS (look-ahead) | Kalman Filter (real-time adaptive) |
| Z-score | Full-sample normalisation | Rolling 252-day (no look-ahead) |
| Pair selection | Manual, 1 pair | Systematic scan of 3,081 pairs |
| Position sizing | Equal-weight | Equal Risk Contribution (ERC) |
| Transaction costs | None | 10 bps one-way |
| Diversification | Not measured | Diversification ratio computed |
| OOS validation | None | Walk-forward (NB09) |

### Diversification ratio > 1.0
When the diversification ratio exceeds 1.0, the portfolio volatility is *lower*
than the average of the individual strategy volatilities — this is the concrete
measure that diversification is working. A ratio of 1.5 means the portfolio
achieves 50% more return-per-unit-of-risk than any single pair in isolation.

### Next steps for productionisation
- **Monthly rebalancing:** recompute ERC weights as the covariance matrix evolves
- **Pair turnover:** add/remove pairs dynamically based on rolling cointegration tests
- **Execution:** account for market impact at NSE lot sizes
- **Risk limits:** add max single-pair drawdown kill switch